In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv(r"E:\Pyhton code\Data Sets\archive\ml-latest-small\ml-latest-small\movies.csv")
ratings = pd.read_csv(r"E:\Pyhton code\Data Sets\archive\ml-latest-small\ml-latest-small\ratings.csv")

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [3]:
user_movie_matrix = ratings.pivot(
    index='movieId',
    columns='userId',
    values='rating'
)

Y = user_movie_matrix.fillna(0).values
R = (user_movie_matrix.notnull()).astype(int).values

print(Y.shape)

(9724, 610)


In [4]:
num_movies, num_users = Y.shape

Y_mean = np.zeros((num_movies, 1))

for i in range(num_movies):
    idx = R[i, :] == 1
    if np.sum(idx) > 0:
        Y_mean[i] = np.mean(Y[i, idx])

Y_norm = (Y - Y_mean) * R

In [5]:
num_features = 10

X = np.random.randn(num_movies, num_features)
W = np.random.randn(num_users, num_features)
b = np.zeros((1, num_users))

In [6]:
def compute_cost(X, W, b, Y, R, lambda_):
    pred = X @ W.T + b
    error = (pred - Y) * R
    
    cost = 0.5 * np.sum(error**2)
    reg = (lambda_/2) * (np.sum(W**2) + np.sum(X**2))
    
    return cost + reg

In [7]:
alpha = 0.001
lambda_ = 1
iterations = 100

for i in range(iterations):
    
    pred = X @ W.T + b
    error = (pred - Y_norm) * R
    
    X_grad = error @ W + lambda_ * X
    W_grad = error.T @ X + lambda_ * W
    b_grad = np.sum(error, axis=0, keepdims=True)
    
    X -= alpha * X_grad
    W -= alpha * W_grad
    b -= alpha * b_grad
    
    if i % 10 == 0:
        print("Iteration", i, "Cost:", compute_cost(X, W, b, Y_norm, R, lambda_))

Iteration 0 Cost: 337569.7311680239
Iteration 10 Cost: 67781485.98044083


C:\Users\SUYASH\AppData\Local\Temp\ipykernel_22032\3826461012.py:7: RuntimeWarning: overflow encountered in matmul
  pred = X @ W.T + b
C:\Users\SUYASH\AppData\Local\Temp\ipykernel_22032\3826461012.py:7: RuntimeWarning: invalid value encountered in matmul
  pred = X @ W.T + b
C:\Users\SUYASH\AppData\Local\Temp\ipykernel_22032\3826461012.py:8: RuntimeWarning: invalid value encountered in multiply
  error = (pred - Y_norm) * R


Iteration 20 Cost: nan
Iteration 30 Cost: nan
Iteration 40 Cost: nan
Iteration 50 Cost: nan
Iteration 60 Cost: nan
Iteration 70 Cost: nan
Iteration 80 Cost: nan
Iteration 90 Cost: nan


In [8]:
pred = X @ W.T + b
pred = pred + Y_mean  # denormalize

In [9]:
user_id = 0

user_pred = pred[:, user_id]

top_indices = np.argsort(user_pred)[::-1][:10]

recommended_movie_ids = user_movie_matrix.index[top_indices]

recommended_movies = movies[movies['movieId'].isin(recommended_movie_ids)]

print(recommended_movies[['title']])

                                                  title
9732                          Gintama: The Movie (2010)
9733  anohana: The Flower We Saw That Day - The Movi...
9734                                Silver Spoon (2014)
9735            Love Live! The School Idol Movie (2015)
9736           Jon Stewart Has Left the Building (2015)
9737          Black Butler: Book of the Atlantic (2017)
9738                       No Game No Life: Zero (2017)
9739                                       Flint (2017)
9740                Bungo Stray Dogs: Dead Apple (2018)
9741                Andrew Dice Clay: Dice Rules (1991)


In [10]:
my_ratings = np.zeros((num_movies, 1))

my_ratings[0] = 5
my_ratings[10] = 4

Y = np.hstack([Y, my_ratings])
R = np.hstack([R, my_ratings != 0])